# FrontAgent Planner — 微调 & 发布

基于 Qwen2.5-Coder-1.5B + LoRA SFT 的前端任务规划模型

**运行环境**: Colab T4 GPU (免费版)

## 0. 准备环境

In [ ]:
!pip install -q unsloth torch transformers datasets accelerate bitsandbytes peft trl huggingface_hub

### 上传项目文件（二选一，取消注释运行）

In [ ]:
import os

USE_GIT = True  # True=git clone, False=上传 zip

if USE_GIT:
    !git clone https://github.com/ceilf6/machine-learning.git
    %cd machine-learning/self-try/frontagent-planner
else:
    from google.colab import files, zipfile
    uploaded = files.upload()
    with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
        z.extractall('.')
    %cd frontagent-planner

print(f"当前目录: {os.getcwd()}")

### 确认当前目录和数据文件

In [ ]:
import os
print(f"当前目录: {os.getcwd()}")
assert os.path.exists('data/train.json'), '找不到 data/train.json，请先确认目录正确'
import json
with open('data/train.json') as f:
    data = json.load(f)
print(f"训练数据: {len(data)} 条")
print(f"示例: {json.dumps(data[0], ensure_ascii=False)[:200]}...")

## 1. 训练

约 100 条数据，T4 上预计 15-30 分钟

In [ ]:
!python train.py --data data/train.json --output output --epochs 3 --lr 2e-4 --batch-size 4

## 2. 评估

用 10 个未见过的前端任务测试微调效果

In [ ]:
!python eval.py --adapter output/lora_adapter --output eval_results.json --compare-base

### 查看评估结果

In [ ]:
import json
with open('eval_results.json') as f:
    results = json.load(f)
s = results['summary']
print(f"JSON 合法率: {s['json_valid_rate']*100:.1f}%")
print(f"完整计划率: {s['success_rate']*100:.1f}%")
print(f"平均步骤数: {s['avg_steps']:.1f}")
print(f"平均延迟:   {s['avg_latency_ms']:.0f}ms")

## 3. 发布到 HuggingFace

### 登录 HuggingFace

In [ ]:
from huggingface_hub import login
# 方式1: 交互式登录
login()

# 方式2: 直接填 token
# login(token="hf_xxxxxxxxxxxxxxxx")

### 推送模型

In [ ]:
!python publish.py --adapter output/lora_adapter --repo-id ceilf6/frontagent-planner-1.5B

发布成功后访问: https://huggingface.co/ceilf6/frontagent-planner-1.5B